# Data Pipeline Test — SPY / EUR-USD / Gold (XAU-USD)

This notebook does two things:

1. **Historical download** — pulls 4 years (2022-08-16 to 2026-08-16) of 5-min, 1-hour, and Daily bars for SPY, EUR/USD, and Gold, and saves each to `data/<market>/<interval>.csv`.
2. **Execution-fetch test** — tests that we can pull a *live/latest* bar or quote for each market the same way the live signal generator eventually will, before we build anything on top of it.

**Data source note:** Alpaca's documented historical endpoints cover stocks, options, crypto, and news — there isn't a documented historical OHLCV bars endpoint for forex in the Trading API (only a `forex/rates` endpoint, not full candles). So the plan here is:

- **SPY** → Alpaca (`StockHistoricalDataClient`)
- **EUR/USD** → Twelve Data (`time_series`)
- **Gold (XAU/USD)** → Twelve Data (`time_series`)

This keeps things simple: Alpaca for equities, Twelve Data for both non-equity markets, one code path for each of the two.

Install deps if needed: `pip install alpaca-py requests pandas python-dotenv --break-system-packages`

## 0. Config

In [ ]:
!pip install alpaca-py requests pandas python-dotenv --break-system-packages

In [ ]:
print('hi')

In [ ]:
import os
import time
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

# --- API keys ---
# Set these as environment variables before launching Jupyter, e.g.:
#   export ALPACA_API_KEY=xxxx
#   export ALPACA_SECRET_KEY=xxxx
#   export TWELVEDATA_API_KEY=xxxx
# or just hardcode them below for local testing (don't commit this file with real keys in it).
ALPACA_API_KEY = os.environ.get("ALPACA_API_KEY", "")
ALPACA_SECRET_KEY = os.environ.get("ALPACA_SECRET_KEY", "")
TWELVEDATA_API_KEY = os.environ.get("TWELVEDATA_API_KEY", "")

assert ALPACA_API_KEY and ALPACA_SECRET_KEY, "Set ALPACA_API_KEY / ALPACA_SECRET_KEY"
assert TWELVEDATA_API_KEY, "Set TWELVEDATA_API_KEY"

# --- Date range ---
START_DATE = datetime(2022, 8, 16, tzinfo=timezone.utc)
END_DATE = datetime(2026, 8, 16, tzinfo=timezone.utc)

# --- Output folder ---
DATA_DIR = Path("data")
for market in ["spy", "eurusd", "gold"]:
    (DATA_DIR / market).mkdir(parents=True, exist_ok=True)

print("Output folders ready:", [str(p) for p in (DATA_DIR / m for m in ["spy", "eurusd", "gold"])])

## 1. Historical data download

### 1.1 SPY — Alpaca

In [ ]:
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest, StockLatestBarRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from alpaca.data.enums import DataFeed

stock_client = StockHistoricalDataClient(ALPACA_API_KEY, ALPACA_SECRET_KEY)

# feed="iex" is free-tier (~2.5% of volume). Switch to DataFeed.SIP if you have Algo Trader Plus.
ALPACA_FEED = DataFeed.IEX

ALPACA_TIMEFRAMES = {
    "5min": TimeFrame(5, TimeFrameUnit.Minute),
    "1h": TimeFrame(1, TimeFrameUnit.Hour),
    "daily": TimeFrame(1, TimeFrameUnit.Day),
}

def fetch_alpaca_bars(symbol: str, timeframe, start: datetime, end: datetime) -> pd.DataFrame:
    request = StockBarsRequest(
        symbol_or_symbols=symbol,
        timeframe=timeframe,
        start=start,
        end=end,
        feed=ALPACA_FEED,
        adjustment="raw",
    )
    bars = stock_client.get_stock_bars(request)
    df = bars.df.reset_index()
    # df has columns: symbol, timestamp, open, high, low, close, volume, trade_count, vwap
    df = df.drop(columns=["symbol"], errors="ignore")
    df = df.rename(columns={"timestamp": "datetime"})
    return df

In [ ]:
for label, tf in ALPACA_TIMEFRAMES.items():
    print(f"Fetching SPY {label} bars from Alpaca...")
    df = fetch_alpaca_bars("SPY", tf, START_DATE, END_DATE)
    out_path = DATA_DIR / "spy" / f"{label}.csv"
    df.to_csv(out_path, index=False)
    print(f"  saved {len(df):,} rows -> {out_path}")
    if not df.empty:
        print(f"  range: {df['datetime'].min()} to {df['datetime'].max()}")

### 1.2 EUR/USD and Gold (XAU/USD) — Twelve Data

In [ ]:
TWELVEDATA_BASE = "https://api.twelvedata.com/time_series"

# interval name -> (twelvedata interval string, chunk size in days per request)
# chunk sizes are conservative so each request stays comfortably under Twelve Data's
# max output size (5000 points/request on most plans). Adjust if you're on a higher tier.
TWELVEDATA_TIMEFRAMES = {
    "5min": ("5min", 14),
    "1h": ("1h", 120),
    "daily": ("1day", 1500),
}

def fetch_twelvedata_chunk(symbol: str, interval: str, start: datetime, end: datetime, retries: int = 3) -> pd.DataFrame:
    params = {
        "symbol": symbol,
        "interval": interval,
        "start_date": start.strftime("%Y-%m-%d %H:%M:%S"),
        "end_date": end.strftime("%Y-%m-%d %H:%M:%S"),
        "timezone": "UTC",
        "outputsize": 5000,
        "apikey": TWELVEDATA_API_KEY,
        "format": "JSON",
    }
    for attempt in range(retries):
        resp = requests.get(TWELVEDATA_BASE, params=params, timeout=30)
        data = resp.json()
        if isinstance(data, dict) and data.get("status") == "error":
            # rate limited -> backoff and retry
            if "limit" in data.get("message", "").lower():
                wait = 8 * (attempt + 1)
                print(f"  rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue
            raise RuntimeError(f"Twelve Data error: {data}")
        values = data.get("values", [])
        if not values:
            return pd.DataFrame(columns=["datetime", "open", "high", "low", "close"])
        df = pd.DataFrame(values)
        df["datetime"] = pd.to_datetime(df["datetime"])
        for col in ["open", "high", "low", "close"]:
            df[col] = df[col].astype(float)
        return df.sort_values("datetime")
    raise RuntimeError(f"Failed to fetch {symbol} {interval} after {retries} retries")

def fetch_twelvedata_series(symbol: str, interval: str, start: datetime, end: datetime, chunk_days: int) -> pd.DataFrame:
    chunks = []
    cursor = start
    while cursor < end:
        chunk_end = min(cursor + pd.Timedelta(days=chunk_days), end)
        print(f"  {symbol} {interval}: {cursor.date()} -> {chunk_end.date()}")
        df = fetch_twelvedata_chunk(symbol, interval, cursor, chunk_end)
        if not df.empty:
            chunks.append(df)
        cursor = chunk_end
        time.sleep(1)  # basic rate-limit courtesy; raise if you're on a higher-throughput plan
    if not chunks:
        return pd.DataFrame(columns=["datetime", "open", "high", "low", "close"])
    full = pd.concat(chunks, ignore_index=True)
    full = full.drop_duplicates(subset="datetime").sort_values("datetime").reset_index(drop=True)
    return full

In [ ]:
for label, (td_interval, chunk_days) in TWELVEDATA_TIMEFRAMES.items():
    print(f"Fetching EUR/USD {label} bars from Twelve Data...")
    df = fetch_twelvedata_series("EUR/USD", td_interval, START_DATE, END_DATE, chunk_days)
    out_path = DATA_DIR / "eurusd" / f"{label}.csv"
    df.to_csv(out_path, index=False)
    print(f"  saved {len(df):,} rows -> {out_path}")
    if not df.empty:
        print(f"  range: {df['datetime'].min()} to {df['datetime'].max()}")

In [ ]:
for label, (td_interval, chunk_days) in TWELVEDATA_TIMEFRAMES.items():
    print(f"Fetching Gold (XAU/USD) {label} bars from Twelve Data...")
    df = fetch_twelvedata_series("XAU/USD", td_interval, START_DATE, END_DATE, chunk_days)
    out_path = DATA_DIR / "gold" / f"{label}.csv"
    df.to_csv(out_path, index=False)
    print(f"  saved {len(df):,} rows -> {out_path}")
    if not df.empty:
        print(f"  range: {df['datetime'].min()} to {df['datetime'].max()}")

## 2. Sanity-check the saved files

In [ ]:
for market in ["spy", "eurusd", "gold"]:
    for label in ["5min", "1h", "daily"]:
        path = DATA_DIR / market / f"{label}.csv"
        if not path.exists():
            print(f"{market}/{label}: MISSING")
            continue
        df = pd.read_csv(path)
        print(f"{market}/{label}: {len(df):,} rows | {df['datetime'].min() if len(df) else 'n/a'} -> {df['datetime'].max() if len(df) else 'n/a'}")
        display(df.head(3))

## 3. Test execution-time (live) data fetching

This is the pattern the live signal generator will use to pull the *most recent* bar/quote — separate from the bulk historical download above. Just confirming each call works and returns something sane before building the serving pipeline on top of it.

### 3.1 SPY — latest bar (Alpaca)

In [ ]:
latest_request = StockLatestBarRequest(symbol_or_symbols="SPY", feed=ALPACA_FEED)
latest_bar = stock_client.get_stock_latest_bar(latest_request)
print(latest_bar["SPY"])

### 3.2 EUR/USD — latest price (Twelve Data)

In [ ]:
resp = requests.get(
    "https://api.twelvedata.com/price",
    params={"symbol": "EUR/USD", "apikey": TWELVEDATA_API_KEY},
    timeout=15,
)
print(resp.json())

### 3.3 Gold (XAU/USD) — latest price (Twelve Data)

In [ ]:
resp = requests.get(
    "https://api.twelvedata.com/price",
    params={"symbol": "XAU/USD", "apikey": TWELVEDATA_API_KEY},
    timeout=15,
)
print(resp.json())

---

**If all three of the section-3 cells return a sane current price/bar with no errors, the fetch layer is confirmed working for both providers before we move to feature engineering.**

Things worth checking once this runs on your machine:
- Row counts look right for the date range (5-min forex/gold bars should noticeably outnumber 5-min SPY bars, since SPY only trades ~6.5h/day vs ~24h/day for forex/gold).
- No large unexplained gaps in the daily files (a few missing days around holidays is expected and fine).
- Twelve Data's free-tier plan may cap how far back intraday history goes and how many requests/minute you get — if you hit `status: error` messages mentioning plan limits, either slow down `time.sleep()` further or check your plan's intraday history limit.